In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

try:
    import catboost as cb
except:
    ! pip install catboost
    import catboost as cb

#### Functions

In [ ]:
def get_catboost_feat_importance(cls_model_inference, pool_eval):
    df_tmp = cls_model_inference.get_feature_importance(
        data=pool_eval,
        type='LossFunctionChange',
        prettified=True,
    )
    df_tmp.columns = ['feature', 'importance']
    df_tmp.sort_values(by='importance', ascending=False, inplace=True)
    return df_tmp

#### Constants

In [ ]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'

str_target = 'has_inst_tag'

list_str_inst = [
    # from ben: 2025-02-21
    'CURRENT',
    'SELF',
    # key words
    'CHIME-STRIDE',
    'CHIMEFINAL',
    # from dustin: 2025-02-24
    'SELF FIN',
    'SELF/LEAD',
    'SELFINC/LEAD',
    'SBNASELFLNDR',
    'SBNA SELF',
    'CHIME',
    'CLEO',
    'CLEO AI',
    'VARO',
    'ATLAS',
    'ATLCAPBKSELF',
    'POSSIBLE',
    'POSSIBLE FIN',
    'KIKOFF',
    'SUPER.COM',
    'STEP',
    'STEP MOBILE',
    'BRIGHT',
    'BRIGHT BLDR',
    'FIG TECH INC',
    'SELF/RENT',
    'SELFBILLSE',
    'PROGRESSRES',
    'FLEX',
    'FLEXFINANCE',
]

#### Output dir

In [ ]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [ ]:
str_filename = 'df.gzip'
str_uri = f's3://20241112-simple-model-test/08_prep_data/{str_filename}'
df = pd.read_parquet(
    str_uri,
)
# sort
df.sort_values(by='request_datetime', ascending=True, inplace=True)
df

#### Create target

In [ ]:
df['list_institutions'] = df['str_institution__tu_pmthx'].apply(
    lambda x: eval(x.replace('nan','None')),
)
df['list_institutions'] = df['list_institutions'].apply(
    lambda x: [] if x is None else x,
)

list_str_col_new = []
for str_inst in tqdm(list_str_inst):
    str_col_new = f'{str_inst}_tag'
    df[str_col_new] = df['list_institutions'].apply(
        lambda x: 1 if str_inst in x else 0,
    )
    list_str_col_new.append(str_col_new)

df['sum'] = df[list_str_col_new].sum(axis=1)
df['has_inst_tag'] = df['sum'].apply(
    lambda x: 1 if x > 0 else 0,
)
flt_mn = df['has_inst_tag'].mean()
print(f'Proportion has tag: {flt_mn:0.4f}')
# show
#df

#### Mark train/test

In [ ]:
df['row'] = list(range(0, df.shape[0]))
df['prop_row'] = df['row'] / df.shape[0]
df['data_set'] = df['prop_row'].apply(
    lambda x: 'train' if x <= 0.8 else 'test',
)
df.drop(['row','prop_row'], axis=1, inplace=True)

#### List cols model

In [ ]:
# rm target
list_cols_model = [col for col in df.columns if col != str_target]
# rm non-numeric
list_cols_object = [col for col in list_cols_model if df[col].dtype not in ['int64','float64']]
list_cols_model = [col for col in df.columns if col not in list_cols_object]
# rm tags
list_cols_model = [col for col in list_cols_model if 'tag' not in col.lower()]
# rm flag
list_cols_model = [col for col in list_cols_model if 'flag' not in col.lower()]
# rm loss
list_cols_model = [col for col in list_cols_model if 'loss' not in col.lower()]
# rm cols to ignore
list_cols_ignore = [
    'data_set',
    'sum',
    'accountid',
    'bigdebtorid__app',
    'days_on_books',
    'bigdebtorid__ln',
    'bigaccountid__ln',
    'strzipcode__app',
    'bitdefault__app',
    'dti__app',
    'pti__app',
    'prop_row',
    'fltapproveddebttoincome__app',
    'fltapprovedapr_contract__app',
    'fltacquisitionfee__app',
    'row',
    'payment__app',
]
list_cols_model = [col for col in list_cols_model if col not in list_cols_ignore]

#### Catboost model

In [ ]:
# get train and test
df_train = df[df['data_set'] == 'train'].copy()
df_test = df[df['data_set'] == 'test'].copy()

# pool data
pool_train = cb.Pool(
    df_train[list_cols_model].copy(), 
    df_train[str_target], 
)
# pool
pool_valid = cb.Pool(
    df_test[list_cols_model].copy(), 
    df_test[str_target], 
)
# init class
cls_model_inference = cb.CatBoostClassifier(
    task_type='CPU',
    nan_mode='Min',
    random_state=42,
    eval_metric='AUC',
    iterations=100,
    learning_rate=None,
    class_weights=None,
    depth=None,
)
# fit
cls_model_inference.fit(
    pool_train,
    eval_set=[pool_valid],
    verbose=True,
    use_best_model=True,
    early_stopping_rounds=10, 
)

#### Get feat importance

In [ ]:
# get importance
df_tmp = get_catboost_feat_importance(
    cls_model_inference=cls_model_inference,
    pool_eval=pool_valid,
)

#### Map description

In [ ]:
# get the data dict
df_data_dict = pd.read_csv('data_dictionary.csv')
dict_map = dict(zip(df_data_dict['feature_name'], df_data_dict['Description']))
df_tmp['description'] = df_tmp['feature'].map(dict_map)

# save
str_filename = 'df_feat_imp.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_tmp.to_csv(str_local_path, index=False)

# show
df_tmp

#### Pivot

In [ ]:
dict_agg = {col: 'mean' for col in df_tmp['feature']}
df_tmp = df.groupby(by=str_target, as_index=False).agg(dict_agg)
dict_map = {
    0: 'No',
    1: 'Yes',
}
df_tmp[str_target] = df_tmp[str_target].map(dict_map)
df_tmp

#### Transpose

In [ ]:
df_tmp_t = df_tmp.set_index(str_target).T
# show
df_tmp_t

#### Map description

In [ ]:
# get the data dict
df_data_dict = pd.read_csv('data_dictionary.csv')
dict_map = dict(zip(df_data_dict['feature_name'], df_data_dict['Description']))
df_tmp_t['tmp'] = df_tmp_t.index
df_tmp_t['description'] = df_tmp_t['tmp'].map(dict_map)
df_tmp_t.drop('tmp', axis=1, inplace=True)
# show
df_tmp_t

#### Save

In [ ]:
str_filename = 'df_feat_imp.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_tmp_t.to_csv(str_local_path, index=True)